# Painel do Fake-Collector-Refiner — lado FALSO

Este notebook **não contém lógica** — só chama as funções de `scripts/web_scraping_limpo.py`
(via `autoreload`: edite o `.py`, rode a célula de novo, já vale).

**Regra de ouro:** nunca cole código de pipeline aqui. Se algo precisa mudar, muda no `.py`.

Etapas deste painel: `main_scraper()` → `main_nlp()`.
O resto do corpus roda por script, no terminal (seção 4).

**Nunca rode o `main_scraper()` e o `main_nlp()` ao mesmo tempo:** os dois reescrevem o `dataset_bruto.json`.

---
## 0. Setup

`import ... as w` (e não `import *`) para o autoreload atualizar também as constantes
(`w.OLLAMA_MODEL`, `w.LIMITE_ARTIGOS_NLP`...).

In [3]:
%load_ext autoreload
%autoreload 2

import json, os, sys
from collections import Counter

# O projeto foi reorganizado em pastas: o módulo vive em scripts/ e os dados em
# datasets/. Sem acrescentar scripts/ ao sys.path, o import falha quando o
# notebook roda a partir da raiz.
_RAIZ = os.path.dirname(os.path.abspath("painel_pipeline.ipynb"))
for sub in ("scripts", "relatorios"):
    caminho = os.path.join(_RAIZ, sub)
    if caminho not in sys.path:
        sys.path.insert(0, caminho)

import web_scraping_limpo as w

print(f"Pasta de trabalho : {os.getcwd()}")
print(f"Modelo configurado: {w.OLLAMA_MODEL}")
print(f"Dataset bruto     : {w.ARQUIVO_BRUTO}")
print(f"   existe? {os.path.exists(w.ARQUIVO_BRUTO)}")

[*] Motor NLP iniciado com Ollama local (qwen3.5:9b).
Pasta de trabalho : c:\Users\filip\Documents\Projetos\Fake-Collector-Refiner
Modelo configurado: qwen3.5:9b
Dataset bruto     : c:\Users\filip\Documents\Projetos\Fake-Collector-Refiner\datasets\dataset_bruto.json
   existe? True


### 0.1 (Opcional) Espelhar a saída num arquivo de log

Útil para execuções longas: grava tudo em `log_execucao.txt` com flush imediato,
então o log sobrevive a um `Ctrl+C` / interrupção do kernel.

**Importante:** o espelho é feito sobre o `sys.stdout` *atual* (o stream do ipykernel,
que escreve na célula) — e não sobre `sys.__stdout__`, que aponta para o terminal onde
o kernel subiu e faria a saída sumir do notebook. A célula desfaz um espelho anterior
antes de criar outro, então pode ser reexecutada à vontade.

Para desligar o espelho, rode a célula 0.2.

In [ ]:
import sys
from datetime import datetime

class _Tee:
    _marca = "tee_painel"   # permite reconhecer um tee antigo mesmo após redefinir a classe
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data); s.flush()
        return len(data)
    def flush(self):
        for s in self.streams: s.flush()
    def isatty(self): return False

# Desfaz um espelho anterior para não empilhar tees a cada reexecução.
if getattr(sys.stdout, "_marca", None) == "tee_painel":
    sys.stdout = sys.stdout.streams[0]

_stdout_notebook = sys.stdout        # stream do ipykernel -> escreve NA CÉLULA
_log = open("log_execucao.txt", "a", encoding="utf-8")
_log.write(f"\n{'='*78}\n### execução iniciada em {datetime.now():%Y-%m-%d %H:%M:%S}\n{'='*78}\n")
sys.stdout = _Tee(_stdout_notebook, _log)

print("[log] saída espelhada em log_execucao.txt — e continua aparecendo aqui na célula.")

### 0.2 Desligar o espelho / recuperar a saída

Se a saída das células sumir, rode isto para voltar ao normal.

In [ ]:
import sys

if getattr(sys.stdout, "_marca", None) == "tee_painel":
    sys.stdout = sys.stdout.streams[0]
    print("[log] espelho desligado — saída normal restaurada.")
else:
    print("[log] nenhum espelho ativo.")

---
## 1. Diagnóstico do Ollama

Rode antes de qualquer etapa que use LLM: servidor no ar, modelo baixado, e uma chamada real respondendo.

In [ ]:
import requests

base = w.OLLAMA_URL.replace("/api/chat", "")

# 1) servidor no ar?
try:
    print("Servidor:", requests.get(base, timeout=5).text.strip())
except Exception as e:
    print(f"[X] Ollama não respondeu em {base} -> {e}")

# 2) modelo baixado?
try:
    tags = requests.get(f"{base}/api/tags", timeout=10).json()
    modelos = sorted(m["name"] for m in tags.get("models", []))
    print("\nModelos disponíveis:")
    for m in modelos:
        print(f"   {'>>' if m == w.OLLAMA_MODEL else '  '} {m}")
    if w.OLLAMA_MODEL not in modelos:
        print(f"\n[!] '{w.OLLAMA_MODEL}' não está na lista. Rode:  ollama pull {w.OLLAMA_MODEL}")
except Exception as e:
    print(f"[X] Falha ao listar modelos: {e}")

# 3) chamada real
print("\nTeste de geração:", (w.chamar_llm("Responda apenas: ok") or "[SEM RESPOSTA]").strip()[:120])

---
## 2. Etapa 1 — Scraper bruto

A coleta do Boatos.org está **encerrada** (13.140 matérias). Só rode para ampliar a base.
Salva a cada 50 matérias; `Ctrl+C` só depois de uma linha `[salvo]`.

In [ ]:
# Varredura EXAUSTIVA: sem limite de matérias, para sozinha no fim da paginação.
# Salva a cada 50 matérias, então pode interromper com Ctrl+C sem perder nada.
w.LIMIT_LINKS_POR_FONTE = None    # None = exaustivo | número = só N novas
w.PAGINA_INICIAL = 1              # ajuste para retomar de onde parou
w.PAUSA_ENTRE_ARTIGOS = 0.3       # cortesia com o site

w.main_scraper()

---
## 3. Etapa 2 — Extração por LLM (`main_nlp`)

Consome os artigos do `dataset_bruto.json` com `processado_pela_ia = False` e grava em
`dataset_tcc_fake_news_final.json`. Artigo que falha continua na fila; pode interromper e retomar.

**Reprocessamento.** Os artigos com defeito já foram devolvidos à fila por
`scripts/marcar_reprocessamento.py`. Aqui só é preciso rodar o `main_nlp()`, que aplica
`scripts/limpeza_viral.py` nas três vias e grava `extracao_versao` com a versão do código.

> ⚠️ **Antes de reprocessar:** `git pull` e **reinicie o kernel** (o autoreload não garante
> que módulos já importados peguem todas as mudanças). Depois rode a célula 0 e confira na
> saída da inspeção (seção 4) que a linha `CÓDIGO: extração ...` mostra a versão mais nova.
> Em 06/10/2026 um reprocessamento rodou com o código anterior sem que nada avisasse.

Faça primeiro o **piloto (3.1)**: não grava nada.


### 3.1 Piloto — 5 artigos da fila, sem gravar
Confira se o viral começa na mensagem (sem "Leia:"/"Versão 1:"), se há um registro por boato
e quanto tempo leva cada artigo.

In [ ]:
import time

N_PILOTO = 5
with open(w.ARQUIVO_BRUTO, encoding="utf-8") as f:
    bruto = json.load(f)
fila = [i for i in bruto if not i.get("processado_pela_ia")]
print(f"fila: {len(fila)} artigos | piloto com {N_PILOTO} ({w.OLLAMA_MODEL})\n")

for art in fila[:N_PILOTO]:
    t0 = time.time()
    boatos = w.extrair_boatos_do_artigo(art)
    print("=" * 78)
    print(f"{art['titulo'][:70]}  ({time.time() - t0:.1f}s)")
    if boatos is None:
        print("   [FALHA] o modelo não devolveu resposta utilizável")
        continue
    if not boatos:
        print("   (nenhum boato transcrito neste artigo)")
    for b in boatos:
        print(f"   [{b['origem_viral']}] literal: {b['verificacao_viral']} | versões: {b['viral_versoes']} "
              f"| prefixos removidos: {b['viral_prefixos_removidos']}")
        print(f"      viral : {b['texto_viral_bruto'][:160]}")
        print(f"      claim : {b['claim_refinada'][:160]}")

fila: 3 artigos | piloto com 5 (qwen3.5:9b)



### 3.2 Execução completa
Estimativa: ~2.600 artigos × 8,6 s ≈ 6 h numa RTX 4060 (menos numa GPU mais forte).

In [2]:
w.LIMITE_ARTIGOS_NLP = None     # ex.: 50 para uma rodada curta
w.main_nlp()

NameError: name 'w' is not defined

---
## 4. Inspeção
Fila restante, origem do viral, literalidade e quantos registros já são da extração v2.

In [1]:
from collections import Counter
from limpeza_viral import limpar_viral

with open(w.ARQUIVO_BRUTO, encoding="utf-8") as f:
    bruto = json.load(f)
with open(w.ARQUIVO_FINAL, encoding="utf-8") as f:
    final = json.load(f)

print(f"CÓDIGO: extração {w.VERSAO_EXTRACAO}  <- confira: precisa ser a versão mais nova do repositório")
print(f"BRUTO : {len(bruto)} matérias | {sum(1 for i in bruto if not i.get('processado_pela_ia'))} na fila")
print(f"FINAL : {len(final)} registros")
print("extração :", dict(Counter(r.get("extracao_versao", "v1") for r in final)))
print("origem   :", dict(Counter(r.get("origem_viral") for r in final)))
print("literal  :", dict(Counter(r.get("verificacao_viral") for r in final)))

# Defeitos que as regras novas eliminam, calculados POR VERSÃO: mostra na hora
# se a extração mais recente rodou com o código certo.
print(f"\n{'versão':14s} {'registros':>9s} {'introdução':>10s} {'versões':>8s} {'degenerado':>10s} {'voz do checador':>15s}")
for versao in sorted({r.get("extracao_versao", "v1") for r in final}):
    regs = [r for r in final if r.get("extracao_versao", "v1") == versao]
    lim = [limpar_viral(r["texto_viral_bruto"]) for r in regs]
    print(f"{versao:14s} {len(regs):9d} {sum(1 for l in lim if l['prefixos']):10d} "
          f"{sum(1 for l in lim if l['versoes'] > 1):8d} {sum(1 for l in lim if l['degenerada']):10d} "
          f"{sum(1 for l in lim if l['checador']):15d}")
print("\nNa versão mais nova, as quatro colunas de defeito devem ser zero.")


ModuleNotFoundError: No module named 'limpeza_viral'

---
## 5. Depois da extração — no terminal (WSL), nesta ordem

Rode pelo Python do WSL (`.venv/bin/python`): no Windows, o Controle Inteligente de Aplicativos
bloqueia DLLs do scikit-learn.

```bash
.venv/bin/python scripts/corrigir_corpus.py --aplicar        # campos, duplicatas, divisão temporal (falso)
.venv/bin/python scripts/preparar_verdadeiras.py --aplicar   # herda os cortes temporais do lado falso
.venv/bin/python scripts/fundir_corpus.py --aplicar          # corpus_fundido.jsonl
.venv/bin/python scripts/modelo_bruto.py                     # e os demais experimentos
```

Os dois primeiros simulam sem `--aplicar` — rode assim antes para conferir.